# P1 full raw-trace replay

Upload this one notebook into `~/P1` and use the existing P1 environment. It reads all four completed full studies and checks their saved raw traces against the compact records reviewed in this chat. **It executes no simulations and edits no experiment data.** Keep both original run directories and all raw files on the server.

The embedded reference fingerprints bind the audit to the two uploaded review ZIPs. It checks protocols, source files, all expected checkpoints, raw/compact hashes, record counts, and exact raw-to-compact agreement through the frozen trace auditors. Full coverage is 11,008 chunks/streams and 1,487,360 recorded executions; about 40 GB compressed raw.

This is a separate replay using existing trace checkers. It is not an independently implemented quantum simulator or validation of physical hardware/deployed attacker access. Six local engineering tests pass. **The production raw replay has not been run here.**


In [ ]:
from pathlib import Path
import sys

PROJECT_DIR = Path.home() / "P1"
MAIN_ROOT = PROJECT_DIR / "feedback-native-20261024-v1"
ADDITIONAL_ROOT = PROJECT_DIR / "additional-diagnostics-20261024-v1"
AUDIT_DIR = PROJECT_DIR / "raw-replay-20261025-v1"
STUDIES = ("timing", "application", "causal", "robustness")
WORKERS = 4
SAMPLE_FIRST = True
RUN_FULL = True
PYTHON_OVERRIDE = None

assert 1 <= WORKERS <= 8
assert STUDIES and len(STUDIES) == len(set(STUDIES))
assert all(s in ("timing","application","causal","robustness") for s in STUDIES)
for root in (MAIN_ROOT.resolve(),ADDITIONAL_ROOT.resolve()):
    assert AUDIT_DIR.resolve() != root and root not in AUDIT_DIR.resolve().parents, "Audit output must be outside the original runs."
PYTHON = str(PYTHON_OVERRIDE or (PROJECT_DIR / ".venv-p1/bin/python" if (PROJECT_DIR / ".venv-p1/bin/python").is_file() else sys.executable))
print({"python":PYTHON,"audit_output":str(AUDIT_DIR),"studies":STUDIES,"workers":WORKERS})


In [ ]:
import base64,hashlib,io,json,os,signal,subprocess,zipfile
PAYLOAD_SHA256 = '3db0229a1d974014790046cf42095b8aef2f037ae1ca29b89c63015ecbb808c8'
PAYLOAD = '...'
blob=base64.b64decode(PAYLOAD)
assert hashlib.sha256(blob).hexdigest()==PAYLOAD_SHA256
TOOL_DIR=PROJECT_DIR / "raw-replay-tool-20261025-v1"
with zipfile.ZipFile(io.BytesIO(blob)) as z:
    for item in z.infolist():
        rel=Path(item.filename)
        if rel.is_absolute() or ".." in rel.parts:raise RuntimeError("Unsafe audit archive path")
        target=TOOL_DIR / rel;body=z.read(item)
        if target.exists() and target.read_bytes()!=body:raise RuntimeError("Existing audit tool differs; preserve it and use a versioned tool directory.")
        target.parent.mkdir(parents=True,exist_ok=True)
        if not target.exists():target.write_bytes(body)
AUDITOR=TOOL_DIR / "auditor"
AUDIT_DIR.mkdir(parents=True,exist_ok=True)
print("Verified audit code:",AUDITOR)


In [ ]:
from IPython.display import FileLink,display

def launch(command):
    command=list(map(str,command));env=dict(os.environ,PYTHONDONTWRITEBYTECODE="1")
    with (AUDIT_DIR / "CONTROLLER.log").open("a",buffering=1) as log:
        log.write("\nCOMMAND "+json.dumps(command)+"\n")
        process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
            text=True,bufsize=1,start_new_session=True,env=env)
        try:
            for line in process.stdout:print(line,end="",flush=True);log.write(line)
            status=process.wait()
        except BaseException:
            try:os.killpg(process.pid,signal.SIGTERM)
            except ProcessLookupError:pass
            try:process.wait(timeout=20)
            except subprocess.TimeoutExpired:
                try:os.killpg(process.pid,signal.SIGKILL)
                except ProcessLookupError:pass
            raise
    if status:raise RuntimeError("Raw audit stopped. Inspect the actual exception above; no incomplete audit is a full PASS.")

launch([PYTHON,AUDITOR / "test_audit.py"])


The sample pass checks first/middle/last chunks in each selected study and is only an integration diagnostic. Its runtime extrapolation is approximate. Set `RUN_FULL=False` to stop after that diagnostic; a sample PASS does not complete the required raw audit.

With the default `RUN_FULL=True`, all selected full studies are then audited. Keep the same audit directory to resume. Every cached PASS requires rehashed unchanged raw/compact files. Four workers are a conservative default for memory and storage I/O; eight are allowed within a suitable allocation. If splitting studies across sessions, choose distinct `AUDIT_DIR` values.


In [ ]:
def command(mode):
    return [PYTHON,AUDITOR / "audit_raw.py","run","--main-root",MAIN_ROOT,
            "--additional-root",ADDITIONAL_ROOT,"--output",AUDIT_DIR,"--mode",mode,
            "--workers",WORKERS,"--studies",*STUDIES]

try:
    if SAMPLE_FIRST:
        launch(command("sample"))
        sample=json.loads((AUDIT_DIR / "sample-REPORT.json").read_text())
        refs=json.loads((AUDITOR / "REVIEWED_REFERENCE.json").read_text())
        seconds=0.
        for study,t in sample["totals"].items():
            seconds += t["worker_seconds"] / t["jobs"] * len(refs[study]["checkpoints"])
        print(f"Rough full audit: {seconds/3600:.2f} serial worker hours, idealized {seconds/3600/WORKERS:.2f} hours at {WORKERS} workers.")
        print("Storage contention and chunk heterogeneity can change this estimate. Sample PASS is not full coverage.")
    if RUN_FULL:launch(command("full"))
finally:
    exported=subprocess.run([PYTHON,str(AUDITOR / "audit_raw.py"),"export","--output",str(AUDIT_DIR)],capture_output=True,text=True)
    print(exported.stdout,exported.stderr)
    report=AUDIT_DIR / "full-REPORT.json"
    if report.exists():print(report.read_text())
    review=AUDIT_DIR.parent / (AUDIT_DIR.name+"-review.zip")
    if review.exists():display(FileLink(os.path.relpath(review,Path.cwd())))
    display(FileLink(os.path.relpath(AUDIT_DIR / "CONTROLLER.log",Path.cwd())))


Return the small audit review ZIP. A complete run covering all four selected studies should report `PASS_FULL_RAW_TO_COMPACT_REPLAY`, with 11,008 jobs and 1,487,360 records. Missing raw files or a mismatch require investigation, not a new simulation run or deletion of unfavorable records. Original study files and historical experiments remain untouched.
